# European Steel Decarbonization
## Temporal Action Score Analysis: Pre-COVID vs Post-COVID

**Team:** SuSteelAible  
**Author:** Irene  
**Date:** December 2024  

---

## Overview

This notebook applies the Action Score framework to two distinct periods:
- **Pre-COVID (2013-2019):** Baseline decarbonization dynamics
- **Post-COVID (2020-2024):** Acceleration period with policy changes

By comparing scores across these periods, we can identify:
- Which companies accelerated decarbonization efforts
- How transformation announcements translated to action
- Whether the policy landscape shift (EU Green Deal, CBAM) impacted trajectories

The analysis culminates in an **animated visualization** showing company movement in the Talk vs Action matrix from 2019 → 2024.

---

## Scoring Methodology

Same framework as main notebook (100 points total):
1. **Performance (30 pts):** Final year intensity for each period (2019 vs 2024)
2. **Trend (30 pts):** Improvement rate within each period
3. **Data Quality (15 pts):** Reporting completeness for each period
4. **Technology (20 pts):** Status as of end of period (2019 vs 2024)
5. **Renewable (0-5 pts):** Renewable % for each period (2019 vs 2024)

**Key difference:** Trend scores calculated separately for each period to capture acceleration/deceleration.

---

Let's begin.

## 1. Setup & Data Loading

In [ ]:
# Setup
import sys
from pathlib import Path

# Add scripts folder to path
scripts_path = Path.cwd().parent / 'scripts'
sys.path.insert(0, str(scripts_path))

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import seaborn as sns
import subprocess

# Import custom modules
from data_loader import (
    load_company_data,
    prepare_analysis_dataset,
    print_data_summary
)

# Import scoring functions
from action_score_functions import (
    calculate_performance_score,
    calculate_trend_score,
    calculate_data_quality_score,
    calculate_renewable_score
)

In [ ]:
# Load data
df_raw = load_company_data(fix_apostrophes=True, filter_region='Europe')
df_analysis = prepare_analysis_dataset(df_raw)
print_data_summary(df_analysis)

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

# Define periods
PRE_PERIOD = (2013, 2019)
POST_PERIOD = (2020, 2024)

companies = sorted(df_analysis['company'].unique())

print(f"\n📊 PERIOD DEFINITIONS:")
print(f"  Pre-COVID: {PRE_PERIOD[0]}-{PRE_PERIOD[1]}")
print(f"  Post-COVID: {POST_PERIOD[0]}-{POST_PERIOD[1]}")
print(f"\n  Companies: {len(companies)}")

## 2. Load Technology Scores

In [ ]:
# Load technology scores
tech_scores = pd.read_excel(
    '../data/emissions_and_production_technology.xlsx',
    sheet_name='technology_scores',
    engine='openpyxl'
)

# Filter to European companies
tech_scores = tech_scores[tech_scores['company'].isin(companies)].copy()

print(f"✓ Technology scores loaded for {len(tech_scores)} companies")
print(f"  Columns: {list(tech_scores.columns)}")

## 3. Calculate Scores for Both Periods

We'll calculate all 5 components for both pre-COVID (2013-2019) and post-COVID (2020-2024) periods.

In [ ]:
def calculate_period_scores(df_raw, df_analysis, tech_scores, companies, period_name, start_year, end_year, renewable_col):
    """
    Calculate all 5 action score components for a specific period.
    
    Parameters
    ----------
    df_raw : pd.DataFrame
        Raw data (for data quality assessment)
    df_analysis : pd.DataFrame
        Prepared analysis dataset
    tech_scores : pd.DataFrame
        Technology scores
    companies : list
        List of company names
    period_name : str
        Period identifier ('pre2020' or 'post2020')
    start_year : int
        Period start year
    end_year : int
        Period end year (used for performance score)
    renewable_col : str
        Column name for renewable % ('renewable_pct_2019' or 'renewable_pct_2024')
    
    Returns
    -------
    pd.DataFrame
        Complete action scores for all companies in this period
    """
    print(f"\n{'='*70}")
    print(f"CALCULATING {period_name.upper()} PERIOD SCORES ({start_year}-{end_year})")
    print(f"{'='*70}")
    
    results = []
    
    for company in companies:
        # Component 1: Performance (final year of period)
        perf = calculate_performance_score(df_analysis, company, year=end_year)
        
        # Component 2: Trend (within period)
        trend = calculate_trend_score(df_analysis, company, start_year=start_year, end_year=end_year)
        
        # Component 3: Data Quality (within period)
        quality = calculate_data_quality_score(df_raw, company, start_year=start_year, end_year=end_year)
        
        # Component 4: Technology (load from Excel - period-specific column)
        tech_row = tech_scores[tech_scores['company'] == company]
        if len(tech_row) > 0:
            # Use period-specific technology score if available
            tech_col = f'technology_score_{end_year}' if f'technology_score_{end_year}' in tech_scores.columns else 'technology_score'
            tech_score = tech_row[tech_col].values[0] if tech_col in tech_row.columns else tech_row['technology_score'].values[0]
        else:
            tech_score = 0
        
        # Component 5: Renewable (period-specific)
        renewable = calculate_renewable_score(company, tech_scores, renewable_col=renewable_col)
        
        # Combine results
        results.append({
            'company': company,
            'period': period_name,
            'performance_score': perf['performance_score'],
            'intensity': perf['intensity'],
            'trend_score': trend['trend_score'],
            'slope_pct': trend.get('slope_pct', np.nan),
            'r_squared': trend.get('r_squared', np.nan),
            'data_quality_score': quality['data_quality_score'],
            'n_years': quality['n_years'],
            'technology_score': tech_score,
            'renewable_score': renewable['renewable_score'],
            'renewable_pct': renewable.get('renewable_pct', np.nan),
            'total_score': (
                perf['performance_score'] +
                trend['trend_score'] +
                quality['data_quality_score'] +
                tech_score +
                renewable['renewable_score']
            )
        })
    
    df_period = pd.DataFrame(results)
    
    print(f"\n✓ Calculated scores for {len(df_period)} companies")
    print(f"  Mean total score: {df_period['total_score'].mean():.1f} / 100")
    print(f"  Score range: {df_period['total_score'].min():.1f} - {df_period['total_score'].max():.1f}")
    
    return df_period

In [ ]:
# Calculate pre-COVID scores (2013-2019)
pre_scores = calculate_period_scores(
    df_raw, df_analysis, tech_scores, companies,
    period_name='pre2020',
    start_year=2013,
    end_year=2019,
    renewable_col='renewable_pct_2019'
)

In [ ]:
# Calculate post-COVID scores (2020-2024)
post_scores = calculate_period_scores(
    df_raw, df_analysis, tech_scores, companies,
    period_name='post2020',
    start_year=2020,
    end_year=2024,
    renewable_col='renewable_pct_2024'
)

## 4. Period Comparison

Compare how companies' scores changed between the two periods.

In [ ]:
# Combine both periods
all_scores = pd.concat([pre_scores, post_scores], ignore_index=True)

# Calculate changes for companies with both periods
comparison = []

for company in companies:
    pre = pre_scores[pre_scores['company'] == company]
    post = post_scores[post_scores['company'] == company]
    
    if len(pre) > 0 and len(post) > 0:
        comparison.append({
            'company': company,
            'pre_total': pre['total_score'].values[0],
            'post_total': post['total_score'].values[0],
            'change': post['total_score'].values[0] - pre['total_score'].values[0],
            'pre_trend': pre['trend_score'].values[0],
            'post_trend': post['trend_score'].values[0],
            'trend_change': post['trend_score'].values[0] - pre['trend_score'].values[0]
        })

comparison_df = pd.DataFrame(comparison).sort_values('change', ascending=False)

print("\n" + "="*70)
print("PERIOD COMPARISON: WHO IMPROVED?")
print("="*70)
print(comparison_df[['company', 'pre_total', 'post_total', 'change']].to_string(index=False))
print("="*70)

## 5. Save Combined Scores

Export the complete temporal dataset for the animation.

In [ ]:
# Save combined scores
all_scores.to_csv('../outputs/action_scores_temporal.csv', index=False)
print("✓ Saved: ../outputs/action_scores_temporal.csv")

# Save comparison
comparison_df.to_csv('../outputs/action_scores_comparison.csv', index=False)
print("✓ Saved: ../outputs/action_scores_comparison.csv")

## 6. Animated Visualization: Talk vs Action (2019 → 2024)

The final visualization shows company movement in the Talk vs Action space from 2019 to 2024.

**Data requirements:**
- Action scores: from this notebook (just calculated)
- Talk scores: from ClimateBERT analysis (separate analysis)

**Note:** If ClimateBERT data is not available, this section will be skipped.

In [ ]:
# Check if ClimateBERT talk scores are available
talk_data_path = Path('../data/climateBERT_output.csv')

if not talk_data_path.exists():
    print("⚠️  ClimateBERT data not found at ../data/climateBERT_output.csv")
    print("   Animation section will be skipped.")
    print("   To generate animation, ensure ClimateBERT analysis is complete.")
    SKIP_ANIMATION = True
else:
    print("✓ ClimateBERT data found")
    SKIP_ANIMATION = False

In [ ]:
if not SKIP_ANIMATION:
    # Check ffmpeg availability
    try:
        subprocess.run(['ffmpeg', '-version'], capture_output=True, check=True)
        print("✓ ffmpeg found")
    except:
        print("\n⚠️  ffmpeg not found. Installing...")
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
        print("✓ ffmpeg installed")

In [ ]:
if not SKIP_ANIMATION:
    # Load talk scores
    talk_df = pd.read_csv(talk_data_path)
    print(f"✓ Loaded Talk Scores: {len(talk_df)} rows")
    
    # Aggregate climate_pct by period
    climate_aggregated = []
    
    for company in talk_df['company'].unique():
        company_data = talk_df[talk_df['company'] == company]
        
        # Pre-COVID
        pre_data = company_data[
            (company_data['year'] >= PRE_PERIOD[0]) & 
            (company_data['year'] <= PRE_PERIOD[1])
        ]
        if len(pre_data) > 0:
            climate_aggregated.append({
                'company': company,
                'period': 'pre2020',
                'climate_pct_mean': pre_data['climate_pct'].mean(),
            })
        
        # Post-COVID
        post_data = company_data[
            (company_data['year'] >= POST_PERIOD[0]) & 
            (company_data['year'] <= POST_PERIOD[1])
        ]
        if len(post_data) > 0:
            climate_aggregated.append({
                'company': company,
                'period': 'post2020',
                'climate_pct_mean': post_data['climate_pct'].mean(),
            })
    
    climate_agg_df = pd.DataFrame(climate_aggregated)
    print(f"✓ Aggregated talk scores for {len(climate_agg_df['company'].unique())} companies")

In [ ]:
if not SKIP_ANIMATION:
    # Company name mapping (for ClimateBERT names)
    name_mapping = {
        'ArcelorMittal': 'ArcelorMittal',
        'SSAB': 'SSAB',
        'Salzgitter AG': 'Salzgitter',
        'Outokumpu': 'Outokumpu',
        'Celsa Group': 'Celsa',
        'SIDENOR Group': 'SIDENOR',
        'Feralpi Group': 'Feralpi',
        'Voestalpine': 'Voestalpine',
        'Acerinox EU': 'Acerinox',
        'Tata Steel Nederland': 'TataSteelNederland',
        'Tata Steel UK': 'TataSteelUK',
        'SHS Group': 'Dillinger',
    }
    
    # Technology mapping
    technology_map = {
        'Celsa': 'EAF',
        'SIDENOR': 'EAF',
        'Feralpi': 'EAF',
        'Outokumpu': 'EAF',
        'Acerinox': 'EAF',
        'SSAB': 'BF-BOF',
        'Salzgitter': 'BF-BOF',
        'TataSteelNederland': 'BF-BOF',
        'TataSteelUK': 'BF-BOF',
        'Dillinger': 'BF-BOF',
        'ArcelorMittal': 'BF-BOF',
        'Voestalpine': 'BF-BOF',
    }
    
    # Colors
    tech_colors = {
        'EAF': '#2E7D32',
        'BF-BOF': '#D25A19'
    }

In [ ]:
if not SKIP_ANIMATION:
    # Prepare animation data
    animation_data = []
    
    for your_name, cb_name in name_mapping.items():
        action_pre = all_scores[
            (all_scores['company'] == your_name) & 
            (all_scores['period'] == 'pre2020')
        ]
        action_post = all_scores[
            (all_scores['company'] == your_name) & 
            (all_scores['period'] == 'post2020')
        ]
        
        climate_pre = climate_agg_df[
            (climate_agg_df['company'] == cb_name) & 
            (climate_agg_df['period'] == 'pre2020')
        ]
        climate_post = climate_agg_df[
            (climate_agg_df['company'] == cb_name) & 
            (climate_agg_df['period'] == 'post2020')
        ]
        
        # 2019 data point
        if len(action_pre) > 0 and len(climate_pre) > 0:
            animation_data.append({
                'company': your_name,
                'company_short': cb_name,
                'year': 2019,
                'action_score': action_pre['total_score'].values[0],
                'talk_score': climate_pre['climate_pct_mean'].values[0],
                'technology': technology_map.get(cb_name, 'Unknown'),
                'color': tech_colors.get(technology_map.get(cb_name, 'Unknown'), 'gray')
            })
        
        # 2024 data point
        if len(action_post) > 0 and len(climate_post) > 0:
            has_2019 = len(action_pre) > 0 and len(climate_pre) > 0
            animation_data.append({
                'company': your_name,
                'company_short': cb_name,
                'year': 2024,
                'action_score': action_post['total_score'].values[0],
                'talk_score': climate_post['climate_pct_mean'].values[0],
                'technology': technology_map.get(cb_name, 'Unknown'),
                'color': tech_colors.get(technology_map.get(cb_name, 'Unknown'), 'gray'),
                'is_new': not has_2019
            })
    
    anim_df = pd.DataFrame(animation_data)
    
    companies_2019 = set(anim_df[anim_df['year'] == 2019]['company_short'].unique())
    companies_2024 = set(anim_df[anim_df['year'] == 2024]['company_short'].unique())
    companies_both = companies_2019 & companies_2024
    companies_new = companies_2024 - companies_2019
    
    print(f"\n✓ Animation data prepared:")
    print(f"  Companies in 2019: {len(companies_2019)}")
    print(f"  Companies in 2024: {len(companies_2024)}")
    print(f"  New reporters: {len(companies_new)}")
    if len(companies_new) > 0:
        print(f"    {', '.join(companies_new)}")

In [ ]:
if not SKIP_ANIMATION:
    # Animation parameters
    TOTAL_FRAMES = 200
    PAUSE_START = 20
    PAUSE_END = 100
    FPS = 15
    
    fig, ax = plt.subplots(figsize=(16, 12))
    
    def smoothstep(t):
        """Smooth easing function"""
        return t * t * (3 - 2 * t)
    
    def update(frame):
        """Update function for animation"""
        ax.clear()
        
        # Calculate progress
        if frame < PAUSE_START:
            progress = 0
            period_label = "2019"
        elif frame >= TOTAL_FRAMES - PAUSE_END:
            progress = 1.0
            period_label = "2024"
        else:
            t = (frame - PAUSE_START) / (TOTAL_FRAMES - PAUSE_START - PAUSE_END)
            progress = smoothstep(t)
            period_label = None
        
        # Get data for each year
        df_2019 = anim_df[anim_df['year'] == 2019]
        df_2024 = anim_df[anim_df['year'] == 2024]
        
        # Plot companies with both years (transitioning)
        for company_short in companies_both:
            data_2019 = df_2019[df_2019['company_short'] == company_short].iloc[0]
            data_2024 = df_2024[df_2024['company_short'] == company_short].iloc[0]
            
            # Interpolate positions
            action_pos = (
                data_2019['action_score'] * (1 - progress) + 
                data_2024['action_score'] * progress
            )
            talk_pos = (
                data_2019['talk_score'] * (1 - progress) + 
                data_2024['talk_score'] * progress
            )
            
            ax.scatter(
                action_pos, talk_pos,
                c=data_2024['color'],
                s=400,
                alpha=0.8,
                edgecolors='black',
                linewidth=2.5,
                zorder=10
            )
            
            ax.annotate(
                company_short,
                (action_pos, talk_pos),
                xytext=(6, 6),
                textcoords='offset points',
                fontsize=12,
                fontweight='bold',
                zorder=11
            )
            
            # Draw arrow at end
            if progress == 1.0:
                distance = np.sqrt(
                    (data_2024['action_score'] - data_2019['action_score'])**2 +
                    (data_2024['talk_score'] - data_2019['talk_score'])**2
                )
                if distance > 5:
                    ax.annotate(
                        '',
                        xy=(data_2024['action_score'], data_2024['talk_score']),
                        xytext=(data_2019['action_score'], data_2019['talk_score']),
                        arrowprops=dict(
                            arrowstyle='->',
                            color=data_2024['color'],
                            lw=2.5,
                            alpha=0.6
                        ),
                        zorder=5
                    )
        
        # Plot new reporters (fade in)
        for company_short in companies_new:
            if progress > 0:
                data_2024 = df_2024[df_2024['company_short'] == company_short].iloc[0]
                
                ax.scatter(
                    data_2024['action_score'],
                    data_2024['talk_score'],
                    c=data_2024['color'],
                    s=400,
                    alpha=0.8 * progress,
                    edgecolors='black',
                    linewidth=2.5,
                    zorder=10
                )
                
                label = company_short + ' *'
                
                ax.annotate(
                    label,
                    (data_2024['action_score'], data_2024['talk_score']),
                    xytext=(6, 6),
                    textcoords='offset points',
                    fontsize=12,
                    fontweight='bold',
                    alpha=progress,
                    zorder=11
                )
        
        # Reference lines
        ax.axhline(50, color='gray', linestyle='--', linewidth=1.5, alpha=0.4, zorder=1)
        ax.axvline(50, color='gray', linestyle='--', linewidth=1.5, alpha=0.4, zorder=1)
        
        # Labels
        ax.set_xlabel('ACTION SCORE (Operational Progress)', 
                     fontsize=18,
                     fontweight='bold')
        ax.set_ylabel('CLIMATE DISCUSSION (% of Report)', 
                     fontsize=18,
                     fontweight='bold')
        
        # Title
        if period_label:
            title_text = f'Talk vs Action — {period_label}'
        else:
            title_text = 'Talk vs Action'
        
        ax.set_title(title_text,
                    fontsize=22,
                    fontweight='bold', 
                    pad=20)
        
        # Legend
        legend_elements = [
            plt.Line2D([0], [0], marker='o', color='w', 
                      markerfacecolor=tech_colors['EAF'], markersize=15,
                      markeredgecolor='black', markeredgewidth=2,
                      label='EAF', linestyle='None'),
            plt.Line2D([0], [0], marker='o', color='w',
                      markerfacecolor=tech_colors['BF-BOF'], markersize=15,
                      markeredgecolor='black', markeredgewidth=2,
                      label='BF-BOF', linestyle='None'),
        ]
        
        if len(companies_new) > 0:
            legend_elements.append(
                plt.Line2D([0], [0], marker='', color='w',
                          label='* = New reporter', linestyle='None')
            )
        
        ax.legend(handles=legend_elements, 
                 loc='upper left', 
                 fontsize=13,
                 framealpha=0.95)
        
        # Grid
        ax.grid(True, alpha=0.3, zorder=0, linewidth=0.8)
        ax.set_axisbelow(True)
        
        # Limits
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        
        # Tick labels
        ax.tick_params(axis='both', labelsize=14)
        
        return []
    
    print(f"\nGenerating animation ({TOTAL_FRAMES} frames at {FPS} FPS)...")
    anim = FuncAnimation(
        fig, update,
        frames=TOTAL_FRAMES,
        interval=1000/FPS,
        blit=False,
        repeat=False
    )

In [ ]:
if not SKIP_ANIMATION:
    # Save as MP4
    print("Saving as MP4 (this may take ~1 minute)...")
    anim.save(
        '../outputs/talk_vs_action.mp4',
        writer='ffmpeg',
        fps=FPS,
        dpi=120,
        codec='libx264',
        bitrate=5000,
        extra_args=['-pix_fmt', 'yuv420p']
    )
    
    print("\n✅ Saved: ../outputs/talk_vs_action.mp4")
    plt.close()
    
    print("\n" + "="*80)
    print("✅ ANIMATION COMPLETE!")
    print("="*80)
    print("\nVideo details:")
    print(f"  Duration: {TOTAL_FRAMES/FPS:.1f} seconds")
    print(f"  Resolution: 1920x1440 (16:12 aspect ratio)")
    print(f"  Format: MP4 (H.264)")
    print(f"  Compatible with: PowerPoint, Keynote, Google Slides")

## Summary

This notebook calculated Action Scores for pre-COVID (2013-2019) and post-COVID (2020-2024) periods, enabling temporal comparison of decarbonization dynamics.

**Key outputs:**
- `action_scores_temporal.csv` - Complete scores for both periods
- `action_scores_comparison.csv` - Period-over-period changes
- `talk_vs_action.mp4` - Animated visualization (if ClimateBERT data available)

**Next steps:**
- Analyze which companies accelerated vs. stagnated
- Correlate score changes with policy interventions
- Use animated matrix in presentation to show temporal dynamics